In [1]:
import json
import os
from pathlib import Path

import polars as pl

from nfl.yahoo_fantasy.api import YahooApiClient, iter_dicts
from nfl.yahoo_fantasy.auth import build_oauth_session, load_token
from nfl.yahoo_fantasy.pipeline import PipelineConfig, run_pipeline, _resolve_weeks
from nfl.yahoo_fantasy.transforms import transform
from nfl.yahoo_fantasy.storage.iceberg import IcebergCatalogConfig

In [3]:
LEAGUE_KEY = "449.l.327657"
SPORT = "nfl"

# Run controls
TARGET_SEASON = 2024
START_WEEK = 1
END_WEEK = 17
USE_CACHE = False
INCLUDE_UNROSTERED_PLAYER_STATS = True

In [4]:
# Project bootstrap and catalog initialization
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise RuntimeError("Cannot locate project root (missing pyproject.toml).")

project_root = find_project_root(Path.cwd())
os.chdir(project_root)

In [5]:
# OAuth session setup
token_path = project_root / ".secrets" / "yahoo_token.json"
credentials_path = project_root / ".secrets" / "credentials.json"
cached_token = load_token(token_path)

file_creds = {}
if credentials_path.exists():
    with open(credentials_path, "r", encoding="utf-8") as f:
        file_creds = json.load(f)

client_id = os.getenv("YAHOO_CLIENT_ID") or str(file_creds.get("client_id", "")).strip()
client_secret = os.getenv("YAHOO_CLIENT_SECRET") or str(file_creds.get("client_secret", "")).strip()
redirect_uri = os.getenv("YAHOO_REDIRECT_URI") or str(file_creds.get("redirect_uri", "")).strip() or "http://localhost:8000"

if not client_id or not client_secret or client_id.startswith("YOUR_") or client_secret.startswith("YOUR_"):
    raise ValueError("Set valid Yahoo OAuth credentials in env vars or .secrets/credentials.json")

oauth_session = build_oauth_session(
    client_id=client_id,
    client_secret=client_secret,
    redirect_uri=redirect_uri,
    token_path=token_path,
    auth_code=os.getenv("YAHOO_AUTH_CODE", ""),
    open_browser=False,
 )

client = YahooApiClient(
    oauth_session=oauth_session,
    use_cache=USE_CACHE,
    validate_contracts=True,
 )

print("OAuth and API client ready")
print(f"  use_cache={USE_CACHE}")
print(f"  token_cached={cached_token is not None}")

OAuth and API client ready
  use_cache=False
  token_cached=True


## View Team Roster Scoring Payloads

In [ ]:
# Advanced debug: probe Yahoo team roster scoring payloads
probe_client = YahooApiClient(
    oauth_session=oauth_session,
    use_cache=False,
    validate_contracts=False,
)

probe_league = probe_client.get_league_metadata(LEAGUE_KEY)
probe_teams = probe_client.get_teams(LEAGUE_KEY)

if not probe_teams:
    print("No teams found for probe.")
else:
    probe_team_key = str(probe_teams[0].get("team_key") or "")
    candidate_weeks = list(range(START_WEEK, END_WEEK + 1))
    probe_payload = None
    probe_week = None
    probe_path = None
    last_error = None

    for week in candidate_weeks:
        for path in [
            f"/team/{probe_team_key}/roster;week={week}/players/stats",
            f"/team/{probe_team_key}/roster;week={week};out=players,stats",
            f"/team/{probe_team_key}/roster;week={week};out=players",
        ]:
            try:
                probe_payload = probe_client.get(path, use_cache=False)
                probe_week = week
                probe_path = path
                break
            except Exception as exc:
                last_error = exc
        if probe_payload is not None:
            break

    if probe_payload is None:
        print("Probe failed")
        print(f"  Last error: {last_error}")
    else:
        player_points_nodes = 0
        player_stats_nodes = 0
        for node in iter_dicts(probe_payload):
            if isinstance(node, dict):
                if "player_points" in node:
                    player_points_nodes += 1
                if "player_stats" in node:
                    player_stats_nodes += 1

        print(f"Probe endpoint: {probe_path}")
        print(f"Probe week: {probe_week}")
        print(f"player_points nodes: {player_points_nodes}")
        print(f"player_stats nodes: {player_stats_nodes}")